# Multi-Agent Reliability Study — Real API Version
## Communication Protocol Efficiency & Failure Propagation

**GR5293 Final Project — OpenAI gpt-4o-mini**

Pipeline: `Orchestrator → Retriever → Analyst → Writer`  
Protocols: `NL` | `JSON` | `SHARED`  
Failure modes: `NONE` | `TOOL_FAIL` | `CONTRADICTION` | `AMBIGUITY` | `RESOURCE_LIMIT`

## 0. Install & Import

In [4]:
!pip install openai statsmodels -q

In [5]:
import json, time, copy, random, hashlib, os
from enum import Enum
from dataclasses import dataclass, field, asdict
from typing import Any, Dict, List, Tuple

import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from openai import OpenAI

print('OK')

OK


## 1. API Key Setup

In [6]:
# Option A — Colab Secrets (recommended)
try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
except Exception:
    # Option B — environment variable or paste directly
    api_key = os.environ.get('OPENAI_API_KEY', 'sk-YOUR-KEY-HERE')

client = OpenAI(api_key=api_key)
MODEL  = 'gpt-4o-mini'

# Sanity check
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': 'Reply with just OK'}],
    max_tokens=5
)
print('API connected:', resp.choices[0].message.content.strip())

API connected: OK


## 2. Enums

In [7]:
class Protocol(str, Enum):
    NL     = 'NL'
    JSON   = 'JSON'
    SHARED = 'SHARED'

class FailureMode(str, Enum):
    NONE           = 'NONE'
    TOOL_FAIL      = 'TOOL_FAIL'
    CONTRADICTION  = 'CONTRADICTION'
    AMBIGUITY      = 'AMBIGUITY'
    RESOURCE_LIMIT = 'RESOURCE_LIMIT'

print('Protocols:', [p.value for p in Protocol])
print('Failures: ', [f.value for f in FailureMode])

Protocols: ['NL', 'JSON', 'SHARED']
Failures:  ['NONE', 'TOOL_FAIL', 'CONTRADICTION', 'AMBIGUITY', 'RESOURCE_LIMIT']


## 3. Communication Logger

In [8]:
@dataclass
class Message:
    run_id:       str
    protocol:     str
    failure_mode: str
    sender:       str
    receiver:     str
    content:      Any
    token_count:  int   = 0    # real token count from API usage
    latency_ms:   float = 0.0
    error_flag:   bool  = False
    timestamp:    float = field(default_factory=time.time)


class CommunicationLogger:
    def __init__(self):
        self.messages: List[Message] = []

    def log(self, msg: Message):
        self.messages.append(msg)

    def summary(self) -> Dict:
        if not self.messages:
            return {}
        err = [m for m in self.messages if m.error_flag]
        return {
            'total_messages':         len(self.messages),
            'total_tokens':           sum(m.token_count  for m in self.messages),
            'total_latency_ms':       round(sum(m.latency_ms for m in self.messages), 2),
            'error_message_count':    len(err),
            'error_propagation_rate': round(len(err) / len(self.messages), 3),
        }

    def clear(self):
        self.messages = []

print('Logger OK')

Logger OK


## 4. LLM Wrapper

In [16]:
# Minimal system prompts — keep short to isolate protocol token differences
SYSTEM_PROMPTS = {
    'orchestrator': 'You are an orchestrator agent. Decompose the given task into subtasks for downstream agents.',
    'retriever':    'You are a data retrieval agent. Return financial data clearly for the analyst.',
    'analyst':      'You are a financial analyst. Analyze the data, highlight key metrics and risks. Flag any data quality issues explicitly.',
    'writer':       'You are a report writer. Write a concise investment report. Note any data caveats if the analysis mentions them.',
}

# JSON protocol: ask model to structure output, but don't enforce strict JSON mode
PROTOCOL_SUFFIX = {
    Protocol.NL:     '',
    Protocol.JSON:   ' Structure your response as JSON with clear field names.',
    Protocol.SHARED: '',
}


def llm_call(role: str, prompt: str, protocol: Protocol,
             temperature: float = 0.3) -> Tuple[str, float, int]:
    """
    Returns (response_text, latency_ms, total_tokens).
    total_tokens is the real count from resp.usage.
    """
    full_prompt = prompt + PROTOCOL_SUFFIX[protocol]
    t0 = time.time()
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPTS[role]},
            {'role': 'user',   'content': full_prompt},
        ],
        temperature=temperature,
        max_tokens=150,
    )
    latency_ms   = round((time.time() - t0) * 1000, 2)
    text         = resp.choices[0].message.content.strip()
    total_tokens = resp.usage.total_tokens
    return text, latency_ms, total_tokens


print('LLM wrapper OK')

LLM wrapper OK


## 5. Mock Financial Data Tool
Simulates an external API. Swap in `akshare` or web search for real data.

In [10]:
APPLE_JSON = {
    'company':            'Apple Inc.',
    'quarter':            'Q1 FY2025',
    'revenue_b':          124.3,
    'revenue_yoy_pct':    4.0,
    'eps':                2.40,
    'eps_yoy_pct':        10.1,
    'net_income_b':       36.3,
    'gross_margin_pct':   46.9,
    'iphone_revenue_b':   69.1,
    'services_revenue_b': 26.3,
}

APPLE_NL = (
    'Apple reported Q1 FY2025 revenue of $124.3B, up 4.0% YoY. '
    'EPS was $2.40, up 10.1% YoY. Net income was $36.3B. '
    'Gross margin was 46.9%. iPhone revenue: $69.1B; Services: $26.3B.'
)


def fetch_data(failure_mode: FailureMode, protocol: Protocol) -> Tuple[Any, bool]:
    """Returns (data, is_error)."""

    if failure_mode == FailureMode.TOOL_FAIL:
        return ({}, True) if protocol != Protocol.NL else ('', True)

    if failure_mode == FailureMode.RESOURCE_LIMIT:
        # Only partial data available
        if protocol == Protocol.NL:
            return ('Apple Q1 FY2025 revenue was $124.3B. Further data unavailable (call limit).', False)
        return ({'company': 'Apple Inc.', 'quarter': 'Q1 FY2025',
                 'revenue_b': 124.3, 'note': 'partial — resource limit reached'}, False)

    # Normal retrieval
    if protocol == Protocol.NL:
        data = APPLE_NL
        if failure_mode == FailureMode.CONTRADICTION:
            data += (' However, a second source reports revenue of only $98.5B, '
                     'directly contradicting the above figure.')
    else:
        data = copy.deepcopy(APPLE_JSON)
        if failure_mode == FailureMode.CONTRADICTION:
            data['revenue_b'] = 98.5
            data['_conflict_note'] = 'Source B reports $98.5B; Source A reports $124.3B'

    return data, False


print('Data tool OK')

Data tool OK


## 6. Shared Memory

In [11]:
class SharedMemory:
    def __init__(self):
        self._state: Dict = {}

    def write(self, agent: str, key: str, value: Any):
        self._state[key] = value

    def read(self, key: str, default=None) -> Any:
        return self._state.get(key, default)

    def overhead_tokens(self) -> int:
        """Extra tokens from holding the full shared state in memory."""
        return max(1, len(json.dumps(self._state, default=str)) // 4)

    def clear(self):
        self._state = {}

print('SharedMemory OK')

SharedMemory OK


## 7. Agent Functions

In [12]:
# ── helpers ───────────────────────────────────────────────────────────────────
ERROR_WORDS_UPSTREAM = ['unavailable', 'no data', 'not found', 'empty', 'failed', 'none']
ERROR_WORDS_DETECT   = ['conflict', 'contradict', 'unavailable', 'missing', 'no data',
                        'inconsistent', 'anomaly', 'warning', 'cannot', 'unable', 'insufficient',
                        'partial', 'incomplete', 'flag']
DEGRADE_WORDS        = ['unavailable', 'incomplete', 'could not', 'unable', 'insufficient',
                        'data issue', 'caution', 'caveat', 'note:', 'however']

def _has_any(text: str, words: List[str]) -> bool:
    t = text.lower()
    return any(w in t for w in words)

def _to_str(data: Any) -> str:
    return json.dumps(data, default=str) if isinstance(data, dict) else str(data)


# ── agents ────────────────────────────────────────────────────────────────────
def agent_orchestrator(task: str, protocol: Protocol, failure_mode: FailureMode,
                       logger: CommunicationLogger, run_id: str) -> Tuple[str, int]:

    if failure_mode == FailureMode.AMBIGUITY:
        task = 'Look into that company and write something useful about its recent performance.'

    prompt = f'Decompose this task for downstream agents: {task}'
    text, ms, tokens = llm_call('orchestrator', prompt, protocol)
    logger.log(Message(run_id=run_id, protocol=protocol.value, failure_mode=failure_mode.value,
                       sender='Orchestrator', receiver='Retriever',
                       content=text, token_count=tokens, latency_ms=ms))
    return text, tokens


def agent_retriever(task_msg: Any, protocol: Protocol, failure_mode: FailureMode,
                    logger: CommunicationLogger, run_id: str) -> Tuple[str, bool, int]:

    raw, is_error = fetch_data(failure_mode, protocol)

    if is_error or not raw:
        prompt = 'The data retrieval tool returned no results. Inform the analyst data is unavailable.'
    else:
        prompt = f'Here is the retrieved financial data: {_to_str(raw)}. Pass it clearly to the analyst.'

    text, ms, tokens = llm_call('retriever', prompt, protocol)
    error_signal = is_error or _has_any(text, ERROR_WORDS_UPSTREAM)

    logger.log(Message(run_id=run_id, protocol=protocol.value, failure_mode=failure_mode.value,
                       sender='Retriever', receiver='Analyst',
                       content=text, token_count=tokens, latency_ms=ms, error_flag=error_signal))
    return text, error_signal, tokens


def agent_analyst(retrieved: Any, protocol: Protocol, failure_mode: FailureMode,
                  logger: CommunicationLogger, run_id: str) -> Tuple[str, bool, int]:

    data_str = _to_str(retrieved)
    prompt = (
        f'Analyze the following financial data. Provide key insights and risks. '
        f'If you notice missing data, contradictions, or anomalies, flag them explicitly. '
        f'Data: {data_str}'
    )
    text, ms, tokens = llm_call('analyst', prompt, protocol)

    error_detected  = _has_any(text, ERROR_WORDS_DETECT)
    upstream_error  = _has_any(data_str, ERROR_WORDS_UPSTREAM)
    propagate       = upstream_error and not error_detected

    logger.log(Message(run_id=run_id, protocol=protocol.value, failure_mode=failure_mode.value,
                       sender='Analyst', receiver='Writer',
                       content=text, token_count=tokens, latency_ms=ms, error_flag=propagate))
    return text, error_detected, tokens


def agent_writer(analysis: Any, protocol: Protocol, failure_mode: FailureMode,
                 logger: CommunicationLogger, run_id: str) -> Tuple[str, bool, int]:

    prompt = (
        f'Write a concise investment report based on this analysis. '
        f'If the analysis flags data quality issues, mention them in the report. '
        f'Analysis: {_to_str(analysis)}'
    )
    text, ms, tokens = llm_call('writer', prompt, protocol)
    report_degraded = _has_any(text, DEGRADE_WORDS)

    logger.log(Message(run_id=run_id, protocol=protocol.value, failure_mode=failure_mode.value,
                       sender='Writer', receiver='Output',
                       content=text, token_count=tokens, latency_ms=ms, error_flag=report_degraded))
    return text, report_degraded, tokens


print('All agents OK')

All agents OK


## 8. Pipeline Runner

In [13]:
@dataclass
class RunResult:
    run_id:                 str
    protocol:               str
    failure_mode:           str
    total_tokens:           int
    total_latency_ms:       float
    total_messages:         int
    error_count:            int
    error_propagation_rate: float
    error_detected:         bool
    report_degraded:        bool
    recovery_success:       bool
    report_snippet:         str


def run_pipeline(protocol: Protocol, failure_mode: FailureMode,
                 task: str = 'Analyze Apple Q1 FY2025 earnings and produce an investment report.',
                 seed: int = 0) -> RunResult:

    random.seed(seed)
    run_id = hashlib.md5(f'{protocol}{failure_mode}{seed}'.encode()).hexdigest()[:8]
    logger = CommunicationLogger()
    shared = SharedMemory()

    if protocol == Protocol.SHARED:
        orch_out, _           = agent_orchestrator(task, protocol, failure_mode, logger, run_id)
        shared.write('Orchestrator', 'task_plan', orch_out)

        task_plan             = shared.read('task_plan', task)
        ret_out, _, _         = agent_retriever(task_plan, protocol, failure_mode, logger, run_id)
        shared.write('Retriever', 'data', ret_out)

        raw                   = shared.read('data', '')
        ana_out, detected, _  = agent_analyst(raw, protocol, failure_mode, logger, run_id)
        shared.write('Analyst', 'analysis', ana_out)

        analysis              = shared.read('analysis', '')
        report, degraded, _   = agent_writer(analysis, protocol, failure_mode, logger, run_id)
        shared.write('Writer', 'report', report)

        extra_tokens = shared.overhead_tokens()

    else:
        orch_out, _           = agent_orchestrator(task, protocol, failure_mode, logger, run_id)
        ret_out, _, _         = agent_retriever(orch_out, protocol, failure_mode, logger, run_id)
        ana_out, detected, _  = agent_analyst(ret_out, protocol, failure_mode, logger, run_id)
        report, degraded, _   = agent_writer(ana_out, protocol, failure_mode, logger, run_id)
        extra_tokens = 0

    s = logger.summary()
    return RunResult(
        run_id=run_id,
        protocol=protocol.value,
        failure_mode=failure_mode.value,
        total_tokens=s.get('total_tokens', 0) + extra_tokens,
        total_latency_ms=s.get('total_latency_ms', 0),
        total_messages=s.get('total_messages', 0),
        error_count=s.get('error_message_count', 0),
        error_propagation_rate=s.get('error_propagation_rate', 0.0),
        error_detected=detected,
        report_degraded=degraded,
        recovery_success=detected and not degraded,
        report_snippet=report[:150],
    )


print('Pipeline runner OK')

Pipeline runner OK


## 9. Cost Estimator — Run Before the Full Grid

In [14]:
N_REPS = 10  # <-- adjust here before running the full grid

N_CELLS      = len(list(Protocol)) * len(list(FailureMode))   # 3 x 5 = 15
N_TOTAL_RUNS = N_CELLS * N_REPS

# gpt-4o-mini pricing (2025): $0.15/1M input, $0.60/1M output
# ~800 input + 400 output tokens per run (4 agents)
est_in  = N_TOTAL_RUNS * 800
est_out = N_TOTAL_RUNS * 400
est_usd = est_in / 1e6 * 0.15 + est_out / 1e6 * 0.60

print(f'Grid:          {len(list(Protocol))} protocols x {len(list(FailureMode))} failure modes x {N_REPS} reps')
print(f'Total runs:    {N_TOTAL_RUNS}')
print(f'Est. tokens:   {est_in+est_out:,}  (input {est_in:,} + output {est_out:,})')
print(f'Est. cost:     ~${est_usd:.3f} USD')

Grid:          3 protocols x 5 failure modes x 10 reps
Total runs:    150
Est. tokens:   180,000  (input 120,000 + output 60,000)
Est. cost:     ~$0.054 USD


## 10. Run Full Experiment Grid
> ⚠️ This cell makes real API calls. Check estimated cost above first.

In [17]:
results = []
total   = N_TOTAL_RUNS
done    = 0

for protocol in Protocol:
    for failure_mode in FailureMode:
        for rep in range(N_REPS):
            try:
                r = run_pipeline(protocol, failure_mode, seed=rep)
                results.append(asdict(r))
            except Exception as e:
                print(f'  ERROR {protocol.value}/{failure_mode.value}/rep{rep}: {e}')
            done += 1
            if done % 15 == 0:
                print(f'  {done}/{total} runs done...')
            time.sleep(0.3)   # rate limit buffer

df = pd.DataFrame(results)
print(f'Done: {len(df)} runs | total tokens: {df["total_tokens"].sum():,}')
df.head()

  15/150 runs done...
  30/150 runs done...
  45/150 runs done...
  60/150 runs done...
  75/150 runs done...
  90/150 runs done...
  105/150 runs done...
  120/150 runs done...
  135/150 runs done...
  150/150 runs done...
Done: 150 runs | total tokens: 194,027


,run_id,protocol,failure_mode,total_tokens,total_latency_ms,total_messages,error_count,error_propagation_rate,error_detected,report_degraded,recovery_success,report_snippet
0,8bacfa3c,NL,NONE,1124,16399.55,4,0,0.0,False,False,False,**Investment Report: Apple Inc. (AAPL) - Q1 FY...
1,ab330206,NL,NONE,1124,14161.29,4,0,0.0,False,False,False,### Investment Report: Apple Inc. (Q1 FY2025)\...
2,73f387d2,NL,NONE,1124,13617.05,4,0,0.0,False,False,False,### Investment Report: Apple Inc. (Q1 FY2025)\...
3,b127a801,NL,NONE,1123,12884.81,4,0,0.0,False,False,False,### Investment Report: Apple Inc. (Q1 FY2025)\...
4,40cedafd,NL,NONE,1128,11657.82,4,0,0.0,False,False,False,### Investment Report: Apple Inc. (Q1 FY2025)\...


## 11. Save Raw Results

In [18]:
df.to_csv('results_raw.csv', index=False)
print('Saved: results_raw.csv')

Saved: results_raw.csv


## 12. RQ1 — Protocol Efficiency (Baseline)

In [19]:
baseline = df[df['failure_mode'] == 'NONE']

eff = baseline.groupby('protocol').agg(
    mean_tokens   = ('total_tokens',     'mean'),
    std_tokens    = ('total_tokens',      'std'),
    mean_latency  = ('total_latency_ms', 'mean'),
    std_latency   = ('total_latency_ms',  'std'),
).round(2)

print('RQ1: Protocol Efficiency — Baseline')
print(eff.to_string())

groups = [baseline[baseline['protocol'] == p]['total_tokens'].values for p in ['NL','JSON','SHARED']]
f, p   = stats.f_oneway(*groups)
print(f'\nOne-way ANOVA: F={f:.3f}, p={p:.4f}', '→ Significant' if p < 0.05 else '→ ns')

RQ1: Protocol Efficiency — Baseline
          mean_tokens  std_tokens  mean_latency  std_latency
protocol                                                    
JSON           1238.0       15.80      10731.41      1047.97
NL             1122.9        3.67      13876.03      1531.44
SHARED         1823.7       19.07      12690.01      2188.53

One-way ANOVA: F=6757.681, p=0.0000 → Significant


## 13. RQ2 — Failure Propagation

In [20]:
failures = df[df['failure_mode'] != 'NONE']

fail_summary = failures.groupby(['protocol', 'failure_mode']).agg(
    detection_rate   = ('error_detected',         'mean'),
    propagation_rate = ('error_propagation_rate', 'mean'),
    recovery_rate    = ('recovery_success',        'mean'),
    degradation_rate = ('report_degraded',         'mean'),
).round(3)

print('RQ2: Failure Propagation — Protocol x Failure Mode')
print(fail_summary.to_string())

try:
    import statsmodels.api as sm
    from statsmodels.formula.api import ols
    model = ols('error_propagation_rate ~ C(protocol) + C(failure_mode) + C(protocol):C(failure_mode)',
                data=failures).fit()
    print('\nTwo-way ANOVA:')
    print(sm.stats.anova_lm(model, typ=2).round(4))
except Exception as e:
    print(f'ANOVA error: {e}')

RQ2: Failure Propagation — Protocol x Failure Mode
                         detection_rate  propagation_rate  recovery_rate  degradation_rate
protocol failure_mode                                                                     
JSON     AMBIGUITY                  0.0             0.000            0.0               0.0
         CONTRADICTION              0.0             0.000            0.0               0.0
         RESOURCE_LIMIT             0.8             0.125            0.4               0.5
         TOOL_FAIL                  0.9             0.525            0.0               1.0
NL       AMBIGUITY                  0.0             0.025            0.0               0.1
         CONTRADICTION              0.4             0.100            0.1               0.4
         RESOURCE_LIMIT             0.3             0.300            0.1               0.5
         TOOL_FAIL                  1.0             0.400            0.4               0.6
SHARED   AMBIGUITY                  0.0

## 14. Bootstrap 95% Confidence Intervals

In [22]:
def bootstrap_ci(data: np.ndarray, n_boot=2000, alpha=0.05) -> Tuple[float, float]:
    boot = [np.mean(np.random.choice(data, size=len(data), replace=True)) for _ in range(n_boot)]
    return round(np.percentile(boot, 100*alpha/2), 3), round(np.percentile(boot, 100*(1-alpha/2)), 3)

print('Token usage (baseline):')
for p in ['NL','JSON','SHARED']:
    d = baseline[baseline['protocol']==p]['total_tokens'].values
    lo, hi = bootstrap_ci(d)
    print(f'  {p:<8} mean={np.mean(d):6.1f}  95% CI [{lo}, {hi}]')

print('\nError detection rate (failure runs):')
for p in ['NL','JSON','SHARED']:
    d = failures[failures['protocol']==p]['error_detected'].astype(float).values
    lo, hi = bootstrap_ci(d)
    print(f'  {p:<8} mean={np.mean(d):.3f}  95% CI [{lo}, {hi}]')

Token usage (baseline):
  NL       mean=1122.9  95% CI [1120.5, 1124.8]
  JSON     mean=1238.0  95% CI [1228.195, 1246.7]
  SHARED   mean=1823.7  95% CI [1812.6, 1836.008]

Error detection rate (failure runs):
  NL       mean=0.425  95% CI [0.275, 0.575]
  JSON     mean=0.425  95% CI [0.275, 0.575]
  SHARED   mean=0.400  95% CI [0.25, 0.55]


## 15. Markov Chain — Error Propagation Model

In [23]:
def build_markov(pdata: pd.DataFrame) -> np.ndarray:
    """States: 0=OK, 1=ERROR_INJECTED, 2=ERROR_DETECTED, 3=ERROR_PROPAGATED"""
    n        = len(pdata)
    p_inject = (pdata['failure_mode'] != 'NONE').sum() / n if n > 0 else 0
    p_detect = pdata['error_detected'].mean()
    p_recov  = pdata['recovery_success'].mean()
    return np.array([
        [1-p_inject, p_inject,   0,          0          ],
        [0,          0,          p_detect,   1-p_detect  ],
        [p_recov,    1-p_recov,  0,          0          ],
        [0,          0,          0,          1          ],
    ])

states = ['OK','ERR_INJECT','ERR_DETECT','ERR_PROP']
print('Markov Transition Matrices\n')
for p in ['NL','JSON','SHARED']:
    T  = build_markov(df[df['protocol']==p])
    sv = np.array([1,0,0,0], dtype=float)
    for _ in range(10):
        sv = sv @ T
    print(f'[{p}]')
    hdr = f'{"":>14}' + ''.join(f'{s:>14}' for s in states)
    print(hdr)
    for i, lbl in enumerate(states):
        print(f'{lbl:>14}' + ''.join(f'{T[i,j]:>14.3f}' for j in range(4)))
    print(f'  P(absorbed -> ERR_PROP after 10 steps): {sv[3]:.3f}\n')

Markov Transition Matrices

[NL]
                          OK    ERR_INJECT    ERR_DETECT      ERR_PROP
            OK         0.200         0.800         0.000         0.000
    ERR_INJECT         0.000         0.000         0.340         0.660
    ERR_DETECT         0.120         0.880         0.000         0.000
      ERR_PROP         0.000         0.000         0.000         1.000
  P(absorbed -> ERR_PROP after 10 steps): 0.989

[JSON]
                          OK    ERR_INJECT    ERR_DETECT      ERR_PROP
            OK         0.200         0.800         0.000         0.000
    ERR_INJECT         0.000         0.000         0.340         0.660
    ERR_DETECT         0.080         0.920         0.000         0.000
      ERR_PROP         0.000         0.000         0.000         1.000
  P(absorbed -> ERR_PROP after 10 steps): 0.991

[SHARED]
                          OK    ERR_INJECT    ERR_DETECT      ERR_PROP
            OK         0.200         0.800         0.000         0.000
 

## 16. Final Summary Table

In [24]:
rows = []
for p in ['NL','JSON','SHARED']:
    b = df[(df['protocol']==p) & (df['failure_mode']=='NONE')]
    f = df[(df['protocol']==p) & (df['failure_mode']!='NONE')]
    rows.append({
        'Protocol':            p,
        'Tokens (baseline)':   round(b['total_tokens'].mean(), 1),
        'Latency ms (base)':   round(b['total_latency_ms'].mean(), 1),
        'Error Detection':     round(f['error_detected'].mean(), 3),
        'Error Propagation':   round(f['error_propagation_rate'].mean(), 3),
        'Recovery Rate':       round(f['recovery_success'].mean(), 3),
        'Report Degradation':  round(f['report_degraded'].mean(), 3),
    })

summary = pd.DataFrame(rows).set_index('Protocol')
print('FINAL SUMMARY')
print(summary.to_string())
summary.to_csv('results_summary.csv')
print('\nSaved: results_summary.csv')
print('\nGuide: Tokens/Latency lower=better | Detection/Recovery higher=better | Propagation lower=better')

FINAL SUMMARY
          Tokens (baseline)  Latency ms (base)  Error Detection  Error Propagation  Recovery Rate  Report Degradation
Protocol                                                                                                             
NL                   1122.9            13876.0            0.425              0.206          0.150               0.400
JSON                 1238.0            10731.4            0.425              0.162          0.100               0.375
SHARED               1823.7            12690.0            0.400              0.181          0.125               0.475

Saved: results_summary.csv

Guide: Tokens/Latency lower=better | Detection/Recovery higher=better | Propagation lower=better


## 17. Sample Report Outputs
Inspect actual LLM-generated reports per condition.

In [25]:
print('Sample report snippets (first rep per condition)\n')
for p in ['NL','JSON','SHARED']:
    for fm in ['NONE','TOOL_FAIL','CONTRADICTION']:
        row = df[(df['protocol']==p) & (df['failure_mode']==fm)]
        if len(row) > 0:
            snippet = row.iloc[0]['report_snippet']
            detected = row.iloc[0]['error_detected']
            print(f'[{p} / {fm}] error_detected={detected}')
            print(f'  {snippet[:130]}')
            print()

Sample report snippets (first rep per condition)

[NL / NONE] error_detected=False
  **Investment Report: Apple Inc. (AAPL) - Q1 FY2025**

**Date:** [Insert Date]

**Overview:**
This report provides an analysis of A

[NL / TOOL_FAIL] error_detected=True
  ### Investment Report

**Date:** [Insert Date]  
**Prepared by:** [Your Name]

#### Overview
This investment report is based on an

[NL / CONTRADICTION] error_detected=False
  ### Investment Report: Apple Inc. (Q1 FY2025)

#### Overview:
This report evaluates Apple's financial performance for the first qu

[JSON / NONE] error_detected=False
  ```json
{
  "investment_report": {
    "company": "Apple Inc.",
    "quarter": "Q1 FY2025",
    "financial_highlights": {
      "t

[JSON / TOOL_FAIL] error_detected=True
  ```json
{
  "investment_report": {
    "date": "2023-10-01",
    "key_insights": [],
    "risks": [],
    "data_quality_issues": [

[JSON / CONTRADICTION] error_detected=False
  ```json
{
  "investment_report": {
    "company"